## 2 Preprocessing

### Load libraries

In [1]:
library(tidyverse)
library(plotrix)
library(RColorBrewer)
library(Hmisc)
library(patchwork)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘Hmisc’


The following objects are masked from ‘package:dplyr’:

    src, summarize


The following objects are masked from ‘package:base’:

    format.pval, units




### Set working directory

In [ ]:
wd <- "/path/to/03_image_analysis/02_chemotaxis_towards_cAMP"
setwd(wd)

### Collect and clean all CSV files exported by TrackMate

In [ ]:
# Define folder containing raw CSVs
csv_folder_path <- file.path(wd, "data/trackmate/csv")

# List all CSV file paths
csv_paths <- list.files(csv_folder_path, pattern = "\\.csv$", full.names = TRUE)

# Read and clean each file into a named list
dataframes_list <- lapply(csv_paths, function(path) {
  base_name <- tools::file_path_sans_ext(basename(path))
  
  df <- read.csv(path, header = TRUE, check.names = TRUE, sep = ",")[-c(1:3), ] %>%
    dplyr::mutate(across(where(~ all(is.na(.) | is.numeric(.))), as.numeric)) %>%  # Convert numeric-like cols safely
    dplyr::mutate(sample = base_name)

  names(df) <- tolower(names(df))
  return(df)
})

# Name the list entries by the cleaned base filenames
names(dataframes_list) <- tools::file_path_sans_ext(basename(csv_paths))

Combine and clean TrackMate *spots* data

In [ ]:
# Use grep to find names with "spots" in them
names_with_spots <- grep("spots", names(dataframes_list), value = TRUE)

# Subset the original list using the names with "spots"
spots_list <- dataframes_list[names_with_spots]

# Combine all dataframes in the list by rows
spots_df <- do.call(rbind, spots_list)

# Reset row names to make them sequential
rownames(spots_df) <- NULL

# Define the columns you want to remove
cols_to_remove <- c("label", "position_z", "position_t", "visibility", "manual_spot_color", "mean_intensity_ch1" , "median_intensity_ch1", "min_intensity_ch1",  "max_intensity_ch1", "total_intensity_ch1", "std_intensity_ch1", "contrast_ch1", "snr_ch1", "contrast_ch2", "snr_ch2")

label,id,track_id,quality,position_x,position_y,position_z,position_t,frame,radius,⋯,ellipse_major,ellipse_minor,ellipse_theta,ellipse_aspectratio,area,perimeter,circularity,solidity,shape_index,sample
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ID400390,400390,0,119,308.14722,1484.339,0,665,665,6.180387,⋯,6.756281943481631,5.982306151376998,0.3534115,1.1293774963233,120.0,40.39023,0.9243543,0.9876543,3.687107,exp1_ko-spots
ID265221,265221,0,213,285.60185,1452.332,0,364,364,8.291860,⋯,10.062124474642026,7.062598611761699,-1.1489524,1.4247056965526919,216.0,55.61380,0.8776037,0.9795918,3.784040,exp1_ko-spots
ID10242,10242,0,279,400.50059,1295.881,0,3,3,9.449123,⋯,10.262066682526301,8.991568617958928,1.1555307,1.1412988232142052,280.5,61.22446,0.9403570,0.9859402,3.655599,exp1_ko-spots
ID246785,246785,0,159,294.84599,1448.196,0,331,331,7.091753,⋯,8.146003917159314,6.628510082008211,-0.8511413,1.2289343783710975,158.0,46.75057,0.9084335,0.9875000,3.719276,exp1_ko-spots
ID317446,317446,0,111,284.73123,1442.110,0,469,469,5.944106,⋯,6.852577309895934,5.468623978633809,1.4026295,1.2530715837602477,111.0,38.41101,0.9454126,1.0000000,3.645812,exp1_ko-spots
ID131079,131079,0,127,361.57682,1161.560,0,168,168,6.383076,⋯,8.274648051718115,5.1091100855067975,1.5600971,1.6195869560906735,128.0,43.11382,0.8653401,0.9922481,3.810759,exp1_ko-spots
ID154631,154631,0,615,322.94335,1207.430,0,197,197,13.940164,⋯,20.339995875760966,10.381829160184818,1.3967707,1.959191926772071,610.5,102.55419,0.7294386,0.9363497,4.150597,exp1_ko-spots
ID418819,418819,0,164,151.34290,1584.701,0,716,716,7.258119,⋯,8.551696506229815,6.506770612059666,0.9653934,1.3142766229348979,165.5,48.23624,0.8938426,0.9792899,3.749509,exp1_ko-spots
ID417805,417805,0,79,175.72222,1565.563,0,712,712,5.030471,⋯,5.962138932464943,5.093424225615518,0.4008740,1.1705561265603093,79.5,35.59455,0.7885149,0.9578313,3.992087,exp1_ko-spots


### Processing data for the spots
Separate spot dataframes, clean up columns, and export as CSV.

In [ ]:
# Extract spot-related tables
names_with_spots <- grep("spots", names(dataframes_list), value = TRUE)
spots_list <- dataframes_list[names_with_spots]

# Combine and clean spot data
spots_df <- do.call(rbind, spots_list)
rownames(spots_df) <- NULL

# Remove unnecessary metadata columns
cols_to_remove <- c(
  "label", "position_z", "position_t", "visibility", "manual_spot_color",
  "mean_intensity_ch1", "median_intensity_ch1", "min_intensity_ch1", 
  "max_intensity_ch1", "total_intensity_ch1", "std_intensity_ch1",
  "contrast_ch1", "snr_ch1", "contrast_ch2", "snr_ch2"
)
spots_df <- spots_df[, !(names(spots_df) %in% cols_to_remove)]

# Reorder and parse sample info
spots_df <- spots_df[, c("sample", setdiff(names(spots_df), "sample"))]
spots_df <- spots_df %>% separate(sample, into = c("exp", "strain"), sep = "[_-]")

# Save cleaned spots dataframe
file_path <- paste0(wd, "/data/trackmate/csv_combined/spots.csv")
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) dir.create(dir_path, recursive = TRUE)
write.csv(spots_df, file_path, row.names = FALSE)


Warning message:
“Expected 2 pieces. Additional pieces discarded in 2554538 rows [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


### Processing data for the edges
Separate edge dataframes, combine, clean, and save as CSV

In [ ]:
# Extract edge-related tables
names_with_edges <- grep("edges", names(dataframes_list), value = TRUE)
edges_list <- dataframes_list[names_with_edges]

# Combine and clean edge data
edges_df <- do.call(rbind, edges_list)
rownames(edges_df) <- NULL

# Remove unnecessary metadata columns
cols_to_remove <- c("label", "edge_z_location", "manual_edge_color")
edges_df <- edges_df[, !(names(edges_df) %in% cols_to_remove)]

# Reorder and parse sample info
edges_df <- edges_df[, c("sample", setdiff(names(edges_df), "sample"))]
edges_df <- edges_df %>% separate(sample, into = c("exp", "strain"), sep = "_")

# Save cleaned edges dataframe
file_path <- paste0(wd, "/data/trackmate/csv_combined/edges.csv")
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) dir.create(dir_path, recursive = TRUE)
write.csv(edges_df, file_path, row.names = FALSE)

### Processing data for the tracks
Separate dataframes of the tracks and combine them, perform some clean up and save as CSV

In [ ]:
# Use grep to find names with "tracks" in them
names_with_tracks <- grep("tracks", names(dataframes_list), value = TRUE)

# Subset the original list using the names with "tracks"
tracks_list <- dataframes_list[names_with_tracks]

# Combine all dataframes in the list by rows
tracks_df <- do.call(rbind, tracks_list)

# Reset row names to make them sequential
rownames(tracks_df) <- NULL

# Define the columns you want to remove
cols_to_remove <- c("label", "track_index", "track_z_location")

# Remove the columns from the dataframe
tracks_df <- tracks_df[, !(names(tracks_df) %in% cols_to_remove)]

# Create a vector of column names with 'sample' first
column_order <- c("sample", setdiff(names(tracks_df), "sample"))

# Reorder the columns based on the vector
tracks_df <- tracks_df[, column_order]

# Separate the 'sample' column into 'exp' and 'strain'
tracks_df <- tracks_df %>% separate(sample, into = c("exp", "strain"), sep = "_")

# Specify the path where you want to save the dataframe
file_path <- paste0(wd, "/data/trackmate/csv_combined/tracks.csv")

# Ensure the output directory exists
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) {
  dir.create(dir_path, recursive = TRUE)
}

# Save the cleaned tracks dataframe
write.csv(tracks_df, file_path, row.names = FALSE)